In [2]:
import json, os, cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [56]:
try:
    with open('test2_results.json', 'r') as file:
        data = json.load(file)
    print("File data =", data)
    
except FileNotFoundError:
    print("Error: The file 'test2_results.json' was not found.")

frames_list = list(data.items())[4][1]
mid_hip_list = []
left_knee_coordinates = []
old_knee_coordinates = (0,0)
for frame in frames_list:
    if not frame['landmarks']:
        mid_hip_list.append(np.nan)  # Append NaN for frames with no landmarks
        left_knee_coordinates.append(old_knee_coordinates)  # Append the last known knee coordinates
        continue
    left_hip = [a for a in frame['landmarks'] if a['name'] == 'LEFT_HIP_frame_reference']
    right_hip = [a for a in frame['landmarks'] if a['name'] == 'RIGHT_HIP_frame_reference']
    left_knee = [a for a in frame['landmarks'] if a['name'] == 'LEFT_KNEE']
    left_knee_coordinates.append((left_knee[0]['x'], left_knee[0]['y']))
    old_knee_coordinates = (left_knee[0]['x'], left_knee[0]['y'])
    if left_hip and right_hip is not []:
        mid_hip_y = (left_hip[0]['y'] + right_hip[0]['y']) / 2
        mid_hip_list.append(mid_hip_y)
        # print(mid_hip_y)
        # print(left_hip, right_hip)

# We interpolate missing values (NaN) in mid_hip_list
mid_hip_list = pd.DataFrame(mid_hip_list).interpolate().values.ravel().tolist()

# Find the derivative of the mid hip y coordinate to find peaks and valleys
window = 30
mid_hip_derivative = np.diff(mid_hip_list)
max_derivative_idx = np.argmax(mid_hip_derivative)
mid_hip_max = np.argmax(mid_hip_list[max_derivative_idx-window:max_derivative_idx+window])
index_of_max = max_derivative_idx - window + mid_hip_max
print("Index of max mid hip y coordinate:", index_of_max)

File data = {'athlete': 'ATH01', 'session': 's1', 'video': 'Pre Cut03.2103582.20240108160826.avi', 'frame_count': 598, 'frames': [{'frame': 0, 'landmarks': [{'index': 11, 'name': 'LEFT_SHOULDER', 'x': 0.1139056459069252, 'y': -0.47081950306892395, 'z': 0.12629759311676025, 'visibility': 0.9998903274536133}, {'index': 12, 'name': 'RIGHT_SHOULDER', 'x': -0.15213735401630402, 'y': -0.4200000762939453, 'z': 0.17297008633613586, 'visibility': 0.9998999834060669}, {'index': 23, 'name': 'LEFT_HIP', 'x': 0.10183383524417877, 'y': -0.008068347349762917, 'z': -0.0020772116258740425, 'visibility': 0.9998866319656372}, {'index': 24, 'name': 'RIGHT_HIP', 'x': -0.1017220988869667, 'y': 0.00810155924409628, 'z': 0.003181588603183627, 'visibility': 0.9998736381530762}, {'index': 25, 'name': 'LEFT_KNEE', 'x': 0.07906674593687057, 'y': 0.3714791536331177, 'z': -0.06463520973920822, 'visibility': 0.8554762005805969}, {'index': 26, 'name': 'RIGHT_KNEE', 'x': -0.10631084442138672, 'y': 0.37901729345321655,

In [58]:
# Get the 'test.csv' file into a pandas dataframe
csv_path = 'test.csv'
try:
    df = pd.read_csv(csv_path)
    print("CSV data =", df.head())
except FileNotFoundError:
    print(f"Error: The file '{csv_path}' was not found.")

CSV data =    frame_id  normal_angle  left_abduction_angle  right_abduction_angle
0         0      0.000000              0.000000               0.000000
1         1     11.066474              3.329304               1.804425
2         2     11.066474              3.329304               1.804425
3         3      3.370075             -3.553595              -1.232215
4         4     37.972139             -1.925841              -0.511088


In [71]:
# trim the video to about 60 frames before and after the index of max mid hip y coordinate
cap = cv2.VideoCapture('annoted.mp4')
window = 40
cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, index_of_max - window))
# Read and display the trimmed video frames
current_frame = index_of_max - window
# Get videos width, height, and fps
width, height, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    x_knee, y_knee = left_knee_coordinates[current_frame]
    cv2.putText(frame, str(round(df['left_abduction_angle'].iloc[current_frame], 2)), (width//2, height//2), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('Trimmed Video', frame)
    current_frame += 1
    cv2.waitKey(100)
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break
    if cap.get(cv2.CAP_PROP_POS_FRAMES) >= index_of_max + window:
        break
cap.release()
cv2.destroyAllWindows()
for i in range (1,5):
    cv2.waitKey(1)
